In [40]:
import torch
import json
import numpy as np
import os

In [14]:
import torch
from transformers import Wav2Vec2Model

class Wav2Vec2Classifier(torch.nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.encoder = Wav2Vec2Model.from_pretrained(
            "facebook/mms-1b-all"
        )
        self.classifier = torch.nn.Linear(
            self.encoder.config.hidden_size,
            num_labels
        )

    def forward(self, input_values):
        outputs = self.encoder(input_values)
        hidden = outputs.last_hidden_state
        pooled = hidden.mean(dim=1)
        logits = self.classifier(pooled)
        return logits

In [15]:
import json

with open("/Users/adithyanv/Documents/SpeakEasy/models/label_map.json") as f:
    label2id = json.load(f)

id2label = {v: k for k, v in label2id.items()}

NUM_LABELS = len(id2label)

print("NUM_LABELS:", NUM_LABELS)

NUM_LABELS: 10


In [22]:
device = "cpu"

In [18]:
test_model = Wav2Vec2Classifier(NUM_LABELS)
test_model.load_state_dict(torch.load("/Users/adithyanv/Documents/SpeakEasy/models/model_final.pt", map_location=device))
test_model.to(device)
test_model.eval()

print("✅ Model reloaded successfully.")

Loading weights: 100%|██████████| 1094/1094 [00:00<00:00, 12204.13it/s]
Wav2Vec2Model LOAD REPORT from: facebook/mms-1b-all
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 
lm_head.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model reloaded successfully.


In [25]:
x = torch.randn(1, 16000)

with torch.no_grad():
    out = test_model(x)

pred = torch.argmax(out, dim=1).item()

print("Prediction ID:", pred)
print("Prediction Label:", id2label[pred])

Prediction ID: 8
Prediction Label: no


In [26]:
pip install sounddevice soundfile


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [33]:
SAMPLE_RATE = 16000
DURATION = 3

def record_audio():
    print("🎤 Speak now...")
    audio = sd.rec(int(DURATION * SAMPLE_RATE),
                   samplerate=SAMPLE_RATE,
                   channels=1)
    sd.wait()
    print("✅ Done")
    return audio.flatten()

def beep(duration=0.3, freq=1000):
    t = np.linspace(0, duration, int(SAMPLE_RATE * duration), False)
    tone = 0.5 * np.sin(2 * np.pi * freq * t)
    sd.play(tone, SAMPLE_RATE)
    sd.wait()

In [34]:
def predict_from_audio_array(audio_array):

    audio_array = audio_array / max(1e-9, np.max(np.abs(audio_array)))

    x = torch.tensor(audio_array, dtype=torch.float32).unsqueeze(0)

    with torch.no_grad():
        out = test_model(x)

    pred = torch.argmax(out, dim=1).item()

    return id2label[pred]

In [35]:
def speak(text):
    print("🔊", text)
    os.system(f'say "{text}"')
    time.sleep(0.5)

In [2]:
BASE = "http://192.168.4.1"

def safe_request(endpoint):
    try:
        requests.get(f"{BASE}{endpoint}", timeout=2)
        print("📡 Sent:", endpoint)
    except:
        print("⚠️ ESP32 not reachable")

def light_on(): safe_request("/light/on")
def light_off(): safe_request("/light/off")
def fan_on(): safe_request("/fan/on")
def fan_off(): safe_request("/fan/off")
def bright_light(): safe_request("/led/bright")
def dim_light(): safe_request("/led/dim")
def led_off(): safe_request("/led/off")
def call_help(): safe_request("/help")
def call_home(): safe_request("/home")

In [3]:
command_actions = {
    "light_on": light_on,
    "light_off": light_off,
    "fan_on": fan_on,
    "fan_off": fan_off,
    "bright_light": bright_light,
    "dim_light": dim_light,
    "led_off": led_off,
    "call_help": call_help,
    "call_home": call_home
}

In [38]:
def confirm_and_execute(max_attempts=3):

    attempts = 0

    while attempts < max_attempts:

        speak("Say your command after beep")
        beep()

        command_audio = record_audio()
        command = predict_from_audio_array(command_audio)

        print("🔮 Command:", command)

        speak(f"I heard {command}. Say yes or no.")
        beep()

        confirm_audio = record_audio()
        response = predict_from_audio_array(confirm_audio)

        print("🧠 Response:", response)

        if response == "yes":
            speak(f"Executing {command}")

            if command in command_actions:
                command_actions[command]()
            else:
                print("⚠️ Unknown command")

            return command

        elif response == "no":
            speak("Retrying")
            attempts += 1

        else:
            speak("Say yes or no clearly")
            attempts += 1

    speak("Cancelled")
    return None

In [50]:
confirm_and_execute()

🔊 Say your command after beep
🎤 Speak now...
✅ Done
🔮 Command: light_off
🔊 I heard light_off. Say yes or no.
🎤 Speak now...
✅ Done
🧠 Response: yes
🔊 Executing light_off
📡 Sent: /light/off


'light_off'

In [8]:
light_off()

📡 Sent: /light/off


In [6]:
light_on()

⚠️ ESP32 not reachable


In [7]:
import requests

BASE = "http://192.168.4.1"

try:
    r = requests.get(BASE, timeout=2)
    print("Connected!", r.status_code)
except Exception as e:
    print("❌ Not reachable:", e)

Connected! 200
